In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE, URL = '/content/code', 'github.com/quocthangtrann/computer_vision_ADAS.git'
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
def sh(cmd):       # chạy lệnh, GOM cả stdout và stderr (stderr của lệnh con không hiện trong ô kết quả Colab nếu không gom)
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
if not os.path.exists(CODE):
    r = sh(f'git clone https://{URL} {CODE}')
    if r.returncode: raise SystemExit('git clone lỗi:\n' + (r.stderr or r.stdout))
else:
    r = sh(f'git -C {CODE} pull --ff-only')
    if r.returncode:       # thường do sửa tay tệp trong /content/code: cất vào stash (KHÔNG xóa) rồi pull lại
        print('git pull lỗi:\n' + (r.stderr or r.stdout).strip()[-800:])
        print('thay đổi cục bộ trong', CODE + ':\n' + (sh(f'git -C {CODE} status --short').stdout.strip() or '(không có)'))
        r = sh(f'git -C {CODE} stash push -u -m deploy-auto && git -C {CODE} pull --ff-only')
        if r.returncode: raise SystemExit('pull vẫn lỗi sau khi stash (không phải do sửa cục bộ):\n' + (r.stderr or r.stdout).strip()[-800:])
        print('đã cất thay đổi cục bộ vào stash (xem lại: git -C /content/code stash list) và pull xong')
subprocess.run(f'rsync -a --exclude .git {CODE}/repo/ /content/drive/MyDrive/capstone_ai/repo/', shell=True, check=True)
try: import segmentation_models_pytorch
except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)
os.chdir('/content/drive/MyDrive/capstone_ai')
print(subprocess.run(f'git -C {CODE} log --oneline -1', shell=True, capture_output=True, text=True).stdout.strip())
print('mã mới nhất đã ở capstone_ai/repo/ (runtime/, config/, tests/ chạy thẳng từ /content/code) | thư mục làm việc:', os.getcwd())


In [ ]:
import os, glob, subprocess
os.chdir('/content/drive/MyDrive/capstone_ai')
W, H = 640, 320
def ready(W=640, H=320):
    # shard HD đã dựng xong (có đủ _x.npy, _y.npy, _ids.csv) trong data/v2/b9_chunks_{W}x{H}/
    out = []
    for p in glob.glob(f'data/v2/b9_chunks*{W}x{H}*/shard_*_ids.csv'):
        s, d = os.path.basename(p)[:-8], os.path.dirname(p)
        if os.path.exists(f'{d}/{s}_x.npy') and os.path.exists(f'{d}/{s}_y.npy'): out.append(s)
    return sorted(out)

for f in ['data/bdd100k_seg.tar', 'data/v2/drv_masks_3274.tar', 'data/v2/pilot_drv_manifest.csv', 'data/v2/manifest_v2_in100k.csv',
          'data/v2/pilot_drv_x_320x160.npy', 'data/v2/pilot_drv_y_320x160.npy', f'data/v2/pilot_drv_x_{W}x{H}.npy', f'data/v2/pilot_drv_y_{W}x{H}.npy']:
    print(f'{f:45s}', f'{os.path.getsize(f) / 1e6:9.0f} MB' if os.path.exists(f) else 'KHÔNG CÓ')
print(f'shard {W}x{H} đã xong:', ready(W, H) or 'chưa có')
print('tiến trình tải shard đang chạy:', subprocess.run('pgrep -af "[b]9_prepare_v2.py"', shell=True, capture_output=True, text=True).stdout.strip() or 'không')
print(subprocess.run('df -h /content /content/drive | tail -2', shell=True, capture_output=True, text=True).stdout)


In [ ]:
import os, shutil, subprocess, sys, numpy as np
os.chdir('/content/drive/MyDrive/capstone_ai')
W, H = 640, 320
# Chạy được trên runtime CPU. Cache thí điểm ở 640x320 (cần cho huấn luyện, cho shard HD và cho int8/eval).
# 1) sao lưu cache 320x160 một lần (để kiểm nó không bị đụng tới)
os.makedirs('data/v2/backup_pilot_320', exist_ok=True)
for f in ['pilot_drv_x_320x160.npy', 'pilot_drv_y_320x160.npy', 'pilot_drv_manifest.csv']:
    dst = f'data/v2/backup_pilot_320/{f}'
    if not os.path.exists(dst): shutil.copyfile(f'data/v2/{f}', dst)
# 2) kiểm chứng trước: dựng lại 320x160 và so từng pixel với cache có sẵn; chỉ dựng 640x320 khi ĐẠT
if os.path.exists(f'data/v2/pilot_drv_x_{W}x{H}.npy') and os.path.exists(f'data/v2/pilot_drv_y_{W}x{H}.npy'):
    print(f'đã có cache thí điểm {W}x{H}, bỏ qua (xóa hai tệp pilot_drv_*_{W}x{H}.npy trên Drive nếu muốn dựng lại)')
else:
    for w, h in [(320, 160), (W, H)]:
        r = subprocess.run([sys.executable, '-u', 'repo/make_hd_pilot.py', '--w', str(w), '--h', str(h)])
        if r.returncode != 0: raise SystemExit(f'make_hd_pilot {w}x{h} lỗi (mã {r.returncode}): dừng, dán đoạn trên cho tôi')
# 3) kiểm sau khi dựng
for f in ['pilot_drv_x_320x160.npy', 'pilot_drv_y_320x160.npy']:
    a = np.load(f'data/v2/{f}', mmap_mode='r'); b = np.load(f'data/v2/backup_pilot_320/{f}', mmap_mode='r')
    print(f, 'cache 320x160 còn nguyên:', a.shape == b.shape and all(np.array_equal(a[i:i + 256], b[i:i + 256]) for i in range(0, len(a), 256)))
for f in [f'pilot_drv_x_{W}x{H}.npy', f'pilot_drv_y_{W}x{H}.npy']:
    a = np.load(f'data/v2/{f}', mmap_mode='r'); print(f, a.shape, a.dtype, f'{os.path.getsize(f"data/v2/{f}") / 1e6:.0f} MB')
v = np.unique(np.asarray(np.load(f'data/v2/pilot_drv_y_{W}x{H}.npy', mmap_mode='r')[:200]))
print('giá trị nhãn trong 200 ảnh đầu:', v.tolist(), '(mong đợi: tập con của 0, 1, 2, 3, 255)')


In [ ]:
# (tùy chọn) kiểm độc lập 24 ảnh: dựng lại từ tar bằng mã viết riêng ở đây, so với cache 640x320
import os, io, shutil, tarfile, numpy as np, pandas as pd
from PIL import Image
os.chdir('/content/drive/MyDrive/capstone_ai')
W, H = 640, 320
os.makedirs('/content/raw', exist_ok=True)
LOC = '/content/raw/bdd100k_seg.tar'
if not os.path.exists(LOC): shutil.copy('data/bdd100k_seg.tar', LOC)
tf = tarfile.open(LOC); mem = {m.name: m for m in tf.getmembers() if m.isfile()}
J = {os.path.basename(n)[:-4]: n for n in mem if n.endswith('.jpg') and '/images_10k/' in n}
S = {os.path.basename(n)[:-13]: n for n in mem if n.endswith('_train_id.png')}
dt = tarfile.open('data/v2/drv_masks_3274.tar'); D = {os.path.basename(m.name)[:-4]: m for m in dt.getmembers() if m.isfile()}
man = pd.read_csv('data/v2/pilot_drv_manifest.csv')
X = np.load(f'data/v2/pilot_drv_x_{W}x{H}.npy', mmap_mode='r'); Y = np.load(f'data/v2/pilot_drv_y_{W}x{H}.npy', mmap_mode='r')
assert len(man) == len(X) == len(Y), (len(man), len(X), len(Y))
CROP = (0, 0, 1280, 600)
idx = np.sort(np.random.default_rng(0).choice(len(man), 24, replace=False)); bx = by = 0
for k in idx:
    i = man.id[k]
    im = Image.open(io.BytesIO(tf.extractfile(mem[J[i]]).read())).convert('RGB')
    s = np.asarray(Image.open(io.BytesIO(tf.extractfile(mem[S[i]]).read())).crop(CROP).resize((W, H), Image.NEAREST))
    d = np.asarray(Image.open(io.BytesIO(dt.extractfile(D[i]).read())).crop(CROP).resize((W, H), Image.NEAREST))
    road, ign = s == 0, s == 255
    lab = road.astype(np.uint8); lab[d == 1] = 2; lab[d == 2] = 3
    lab[((d == 1) | (d == 2)) & ~road & ~ign] = 255; lab[ign] = 255
    bx += int((np.asarray(im.crop(CROP).resize((W, H), Image.BILINEAR)) != np.asarray(X[k])).sum()); by += int((lab != np.asarray(Y[k])).sum())
print(f'{len(idx)} ảnh | pixel ảnh lệch: {bx} | pixel nhãn lệch: {by}')
print('ĐẠT' if bx == 0 and by == 0 else 'LỆCH: dừng, dán kết quả cho tôi')


In [ ]:
import os, sys, glob, subprocess
from google.colab import userdata
os.chdir('/content/drive/MyDrive/capstone_ai'); os.makedirs('runs', exist_ok=True)
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
W, H = 640, 320
N_SHARDS = 4          # số shard 640x320 cần có (mỗi shard khoảng 6 GB trên Drive); shard đã xong được bỏ qua
FOREGROUND = False    # False: chạy nền (cell trả về ngay, theo dõi bằng cell dưới); True: chạy và in log tại đây
def ready(W=640, H=320):
    # shard HD đã dựng xong (có đủ _x.npy, _y.npy, _ids.csv) trong data/v2/b9_chunks_{W}x{H}/
    out = []
    for p in glob.glob(f'data/v2/b9_chunks*{W}x{H}*/shard_*_ids.csv'):
        s, d = os.path.basename(p)[:-8], os.path.dirname(p)
        if os.path.exists(f'{d}/{s}_x.npy') and os.path.exists(f'{d}/{s}_y.npy'): out.append(s)
    return sorted(out)

LOG = f'runs/b9_prepare_{W}x{H}.log'
assert os.path.exists(f'data/v2/pilot_drv_y_{W}x{H}.npy'), 'cần cache thí điểm 640x320 (cell trên) trước khi dựng shard HD'
running = subprocess.run('pgrep -af "[b]9_prepare_v2.py"', shell=True, capture_output=True, text=True).stdout.strip()
cmd = [sys.executable, '-u', 'repo/b9_prepare_v2.py', '--shards', str(N_SHARDS), '--w', str(W), '--h', str(H)]
if len(ready(W, H)) >= N_SHARDS: print(f'đã đủ {N_SHARDS} shard:', ready(W, H))
elif running: print('đang có tiến trình tải chạy nền, không chạy thêm:', running)
elif FOREGROUND:
    with open(LOG, 'a') as lg:
        pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for ln in pr.stdout: lg.write(ln); print(ln, end='', flush=True)
    if pr.wait() != 0: raise SystemExit(f'b9_prepare_v2 lỗi: dán đoạn log trên cho tôi')
else:
    DL = subprocess.Popen(cmd, stdout=open(LOG, 'a'), stderr=subprocess.STDOUT)
    print(f'đã chạy nền tải và dựng shard {W}x{H} (pid {DL.pid}, log {LOG}). Phiên Colab ngắt thì chạy lại cell này: shard đã xong được bỏ qua.')


In [ ]:
import os, glob, subprocess
os.chdir('/content/drive/MyDrive/capstone_ai')
W, H = 640, 320
def ready(W=640, H=320):
    # shard HD đã dựng xong (có đủ _x.npy, _y.npy, _ids.csv) trong data/v2/b9_chunks_{W}x{H}/
    out = []
    for p in glob.glob(f'data/v2/b9_chunks*{W}x{H}*/shard_*_ids.csv'):
        s, d = os.path.basename(p)[:-8], os.path.dirname(p)
        if os.path.exists(f'{d}/{s}_x.npy') and os.path.exists(f'{d}/{s}_y.npy'): out.append(s)
    return sorted(out)

print('tiến trình:', subprocess.run('pgrep -af "[b]9_prepare_v2.py"', shell=True, capture_output=True, text=True).stdout.strip() or 'không chạy')
print(f'shard {W}x{H} đã xong:', ready(W, H) or 'chưa có')
print(subprocess.run(f'tail -5 runs/b9_prepare_{W}x{H}.log', shell=True, capture_output=True, text=True).stdout)
print(subprocess.run('df -h /content /content/drive | tail -2', shell=True, capture_output=True, text=True).stdout)
